## 파인튜닝에 맞는 데이터 형태 변환

In [1]:
!pip install openai

  Using cached openai-1.64.0-py3-none-any.whl.metadata (27 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.7-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.14.0-py3-none-any.whl.metadata (8.2 kB)
Using cached openai-1.64.0-py3-none-any.whl (472 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.7-py3-none-any.whl (78 kB)
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 74.9 MB/s eta 0:00:00
Using cached h11-0.14.0-py3-none-any.whl (58 kB)


In [2]:
pip install langchain_community langchain_openai faiss-cpu pypdf

   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 74.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/13.7 MB ? eta -:--:--
   --------------------------- ------------ 9.4/13.7 MB 104.5 MB/s eta 0:00:01
   -------------------------------------- - 13.1/13.7 MB 32.2 MB/s eta 0:00:01
   ---------------------------------------- 13.7/13.7 MB 30.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 69.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   -------------------- ------------------- 6.6/12.6 MB 37.0 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 31.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 83.8 MB/s eta 0:00:00
   -------------------------

In [3]:
!pip install python-dotenv
import os 
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv()

# 환경 변수 가져오기
API_KEY = os.getenv("API_KEY")

In [8]:
import json
import os
import re
from langchain.vectorstores import FAISS
from langchain.embeddings.openai import OpenAIEmbeddings

# JSON 파일 로드
json_file_path = "./data_preprocessed/지문_주제별분류/dataset50/validationset.json"
with open(json_file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

# FAISS 인덱스 로드
faiss_passage_all_path = "./faiss_index/faiss_index_passage_all"

embeddings_model = OpenAIEmbeddings(openai_api_key=API_KEY)

vector_db_passage_all = FAISS.load_local(faiss_passage_all_path, embeddings_model, allow_dangerous_deserialization=True)


# FAISS 기반 문서 검색 함수
def search_relevant_passages(subject, topics):
    query = ", ".join([subject] + topics)
    results = vector_db_passage_all.similarity_search(query, k=1)

    return results

# 텍스트에서 개행 및 중복 공백 제거 함수
def clean_text(text):
    return re.sub(r'\s+', ' ', text).strip()
    
# 데이터 변환 함수
def transform_grouped_data(data):
    transformed_data = []

    for entry in data:
        passage = entry["passage"] 
        subject = entry["subject"]
        topics = entry["topics"] 

        # RAG 기반 관련 문서 검색
        relevant_passages = search_relevant_passages(subject, topics)
        passage_guidelines = "".join([doc.page_content for doc in relevant_passages])

        transformed_entry = {
            "messages": [
                {
                    "role": "system",
                    "content": clean_text(
                        '''
                        당신은 대한민국 수학능력시험 국어영역 독서 과목의 지문을 출제하는 한국교육과정평가원 출제위원이다. 
                        고등학교 3학년 수준의 수험생을 평가할 수 있는 지문을 아래의 핵심 논점 및 난이도 요구사항, 작성 조건, 금지사항을 반영하여 작성하십시오.

                        **핵심 논점 및 난이도 요구사항**
                        - 주요 개념 간의 관계를 논리적으로 설명하고, 지문에 나타난 논지의 타당성을 검토할 것
                        - 동일한 화제에 대한 상반되거나 다양한 관점을 비교·분석하며, 각 관점의 타당성을 비판적으로 평가할 수 있도록 구성할 것
                        - 각 분야의 학문적 배경을 반영하되, 개념적 깊이를 확보하고, 전문 용어는 문맥 속에서 명확히 설명할 것
                        - 단순한 정보 전달이 아니라 수험생이 논리적 추론을 수행할 수 있도록 유도할 것

                        **작성 조건**  
                        -문장당 어절 수: 평균 17~25어절을 유지하되, 자연스러운 문장 흐름을 고려할 것
                        - 글자 수: 한글 기준(공백 포함) 최소 1200자, 최대 2200자
                        - 문체: 문어체 사용, 종결어미는 반드시 ‘-다.’ 사용, 문장은 반드시 완결성을 갖출 것
                        - 개념 설명 방식: 장·단점 나열 방식이 아닌 개념 간 관계 중심 설명
                        - 표현 반복 금지: 같은 단어 반복 대신 유의어나 대체어 활용
                        - 문법 준수: 맞춤법, 띄어쓰기, 주어-서술어 호응 고려
                        - 신뢰성 준수 : 일정한 기준을 유지하여 신뢰할 수 있는 결과 제공
                        - 인용 표기: 인용 문장은 “ ”, 인용된 어구는 ‘ ’ 사용

                        **금지 사항** 
                        - 모호하거나 중의적인 표현 사용 금지
                        - 문학 작품 생성 금지
                        - 허구적 사건이나 인물명 사용 금지
                        - 비문 생성 금지 - (가), (나), (다) 등의 기호 사용 금지
                        - 결론에서 전체 내용을 요약하거나 교훈 제시 금지
                        - 자극적이거나 선정적인 문체 사용 금지
                        - 특정 집단을 비하하거나 옹호하는 내용 또는 잘못된 고정관념을 유발하는 내용 작성 금지

                        위 조건을 철저히 준수하여 수학능력시험 국어영역 독서 과목 지문을 출제하십시오.  
                        '''
                    ),
                },
                {
                    "role": "user",
                    "content": clean_text(
                        f'''
                        다음 {subject}과 {",".join(topics)}를 바탕으로 대한민국 수학능력시험 국어영역 독서 과목 문제 풀이에 적합한 지문을 작성하세요.  
                        {subject} 분야에서 {",".join(topics)}을 핵심 제재로 활용하여 논리적이고 구조적인 지문을 구성하십시오.  

                        [출제 기준]  
                        아래 {subject} 분야의 출제 경향 및 작성 원칙을 충실히 반영하여 지문을 작성하십시오.  

                        {passage_guidelines}  

                        **반영 방법**  
                        - {subject} 분야의 글이란? : 해당 분야의 글의 본질적인 특성과 주요 논의 대상을 명확히 포함할 것.  
                        - {subject} 분야의 글 읽기 방법 :독자가 글의 구조와 전개 방식을 쉽게 이해할 수 있도록 논지를 명확히 전달할 것.  
                        - {subject} 분야의 출제 경향 : 기존 출제 방식과 논리적 구성 원칙을 반영하여 지문을 구성할 것.  

                        이 기준을 기반으로, 독자가 개념을 명확히 이해하고 논리적 추론을 수행할 수 있도록 논리적인 전개 방식과 구조를 갖춘 지문을 작성하십시오.  
                        실제 수능 독서 지문과 유사한 형식과 난이도를 유지하십시오. '''
                    ),
                },
                {"role": "assistant", "content": passage}
            ]
        }

        transformed_data.append(transformed_entry)

    return transformed_data


# 데이터 변환
converted_data = transform_grouped_data(data)

# 변환된 데이터 저장 (JSONL 형식)
output_file_path = "./파인튜닝용/2차파인튜닝용/validationset.jsonl"
with open(output_file_path, "w", encoding="utf-8") as outfile:
    for entry in converted_data:
        json.dump(entry, outfile, ensure_ascii=False)
        outfile.write("\n")

print(f"변환된 JSONL 파일이 저장되었습니다: {output_file_path}")



변환된 JSONL 파일이 저장되었습니다: ./파인튜닝용/2차파인튜닝용/validationset.jsonl


In [15]:
import json
import os
import re
from langchain.vectorstores import FAISS
from langchain.embeddings.openai import OpenAIEmbeddings

# JSON 파일 로드
json_file_path = "./data_preprocessed/지문_주제별분류/dataset50/validationset.json"
with open(json_file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

# FAISS 인덱스 로드
faiss_passage_all_path = "./faiss_index/faiss_index_passage_all"

embeddings_model = OpenAIEmbeddings(openai_api_key=API_KEY)

vector_db_passage_all = FAISS.load_local(faiss_passage_all_path, embeddings_model, allow_dangerous_deserialization=True)


# FAISS 기반 문서 검색 함수
def search_relevant_passages(subject, topics):
    query = ", ".join([subject] + topics)
    results = vector_db_passage_all.similarity_search(query, k=1)

    return results

# 텍스트에서 개행 및 중복 공백 제거 함수
def clean_text(text):
    return re.sub(r'\s+', ' ', text).strip()
    
# 데이터 변환 함수
def transform_grouped_data(data):
    transformed_data = []

    for entry in data:
        passage = entry["passage"] 
        subject = entry["subject"]
        topics = entry["topics"] 

        # RAG 기반 관련 문서 검색
        relevant_passages = search_relevant_passages(subject, topics)
        passage_guidelines = "".join([doc.page_content for doc in relevant_passages])

        transformed_entry = {
            "messages": [
                {
                    "role": "system",
                    "content": clean_text(
                        '''
                        당신은 대한민국 수학능력시험 국어영역 독서 과목의 지문을 출제하는 한국교육과정평가원 출제위원이다. 
                        고등학교 3학년 수준의 수험생을 평가할 수 있는 지문을 아래의 핵심 논점 및 난이도 요구사항, 작성 조건, 금지사항을 반영하여 작성하십시오.

                        **핵심 논점 및 난이도 요구사항**
                        - 주요 개념 간의 관계를 논리적으로 설명하고, 지문에 나타난 논지의 타당성을 검토할 것
                        - 동일한 화제에 대한 상반되거나 다양한 관점을 비교·분석하며, 각 관점의 타당성을 비판적으로 평가할 수 있도록 구성할 것
                        - 각 분야의 학문적 배경을 반영하되, 개념적 깊이를 확보하고, 전문 용어는 문맥 속에서 명확히 설명할 것
                        - 단순한 정보 전달이 아니라 수험생이 논리적 추론을 수행할 수 있도록 유도할 것

                        **작성 조건**  
                        -문장당 어절 수: 평균 17~25어절을 유지하되, 자연스러운 문장 흐름을 고려할 것
                        - 글자 수: 한글 기준(공백 포함) 최소 1200자, 최대 2200자
                        - 문체: 문어체 사용, 종결어미는 반드시 ‘-다.’ 사용, 문장은 반드시 완결성을 갖출 것
                        - 개념 설명 방식: 장·단점 나열 방식이 아닌 개념 간 관계 중심 설명
                        - 표현 반복 금지: 같은 단어 반복 대신 유의어나 대체어 활용
                        - 문법 준수: 맞춤법, 띄어쓰기, 주어-서술어 호응 고려
                        - 신뢰성 준수 : 일정한 기준을 유지하여 신뢰할 수 있는 결과 제공
                        - 인용 표기: 인용 문장은 “ ”, 인용된 어구는 ‘ ’ 사용

                        **금지 사항** 
                        - 모호하거나 중의적인 표현 사용 금지
                        - 문학 작품 생성 금지
                        - 허구적 사건이나 인물명 사용 금지
                        - 비문 생성 금지 - (가), (나), (다) 등의 기호 사용 금지
                        - 결론에서 전체 내용을 요약하거나 교훈 제시 금지
                        - 자극적이거나 선정적인 문체 사용 금지
                        - 특정 집단을 비하하거나 옹호하는 내용 또는 잘못된 고정관념을 유발하는 내용 작성 금지

                        위 조건을 철저히 준수하여 수학능력시험 국어영역 독서 과목 지문을 출제하십시오.  
                        '''
                    ),
                },
                {
                    "role": "user",
                    "content": clean_text(
                        f'''
                        다음 {subject}과 {",".join(topics)}를 바탕으로 대한민국 수학능력시험 국어영역 독서 과목 문제 풀이에 적합한 지문을 작성하세요.  
                        {subject} 분야에서 {",".join(topics)}을 핵심 제재로 활용하여 논리적이고 구조적인 지문을 구성하십시오.  
                        독자가 개념을 명확히 이해하고 논리적 추론을 수행할 수 있도록 논리적인 전개 방식과 구조를 갖춘 지문을 작성하십시오.  
                        실제 수능 독서 지문과 유사한 형식과 난이도를 유지하십시오. '''
                    ),
                },
                {"role": "assistant", "content": passage}
            ]
        }

        transformed_data.append(transformed_entry)

    return transformed_data


# 데이터 변환
converted_data = transform_grouped_data(data)

# 변환된 데이터 저장 (JSONL 형식)
output_file_path = "./파인튜닝용/2차파인튜닝용_RAG제거/validationset.jsonl"
with open(output_file_path, "w", encoding="utf-8") as outfile:
    for entry in converted_data:
        json.dump(entry, outfile, ensure_ascii=False)
        outfile.write("\n")

print(f"변환된 JSONL 파일이 저장되었습니다: {output_file_path}")


변환된 JSONL 파일이 저장되었습니다: ./파인튜닝용/2차파인튜닝용_RAG제거/validationset.jsonl
